# Week 2 Lab — Stationarity, the ACF, and Ergodicity
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University

---

### What this lab does

Week 2 said: *stationarity fixes the rules, the ACF describes the dependence, and ergodicity is what makes
one observed path enough.* This notebook makes that concrete.

| Step | What you do |
|---|---|
| 1 | Simulate white noise, AR(1) and a random walk with one fixed seed |
| 2 | Look at the three paths — which one has "rules that do not change"? |
| 3 | Implement the sample ACF **from the formula**, then check it against `statsmodels` |
| 4 | Average along one path vs across many paths — ergodic, and not ergodic |
| 5 | Measure what positive autocorrelation costs: `Var(x̄)` at `φ = 0.6` vs `φ = 0.95` |
| 6 | Your turn — four small changes, two sentences each |

### How to use it

Two cells are marked **`TODO`**. Write those yourself first — they are a couple of lines each and they are the
part worth doing by hand. 직접 채워 보는 것이 이 노트북의 핵심입니다.

> Fill the indicated lines, **remove the entire `raise NotImplementedError(...)` block**, and include the return statement.
> TODO를 채운 뒤 `raise NotImplementedError(...)` 블록 전체를 지우고 `return`까지 작성하세요.

> **Before you type anything: File ▸ Save a copy in Drive.** The link opens read-only, so edits are lost unless you are working in your own copy.
> **타이핑 전에 먼저 `파일 ▸ Drive에 사본 저장`.** 링크는 읽기 전용으로 열리므로, 사본을 만들지 않으면 작성한 코드가 저장되지 않습니다.

> An unfilled `TODO` cell raises `NotImplementedError`. **That is expected, not a broken notebook.**
> Each `TODO` is followed by a collapsed **Solution (정답)** cell. If stuck, run it to continue. If your own function works, **skip the Solution cell**: it overwrites your function.
> 직접 완성했다면 정답 셀은 건너뛰세요. 정답 셀을 실행하면 작성한 함수가 덮어써집니다.

> This week compares basic ACF shapes. Formal model identification and uncertainty bands are Week 6 topics. The familiar `±2/√n` band is a pointwise white-noise approximation, not a universal band for every process.

## 0. Setup

Nothing to install this week — everything below ships with Colab. Later notebooks add an install cell here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

SEED = 42                       # fixed seed = reproducible; every graded submission needs one

plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("numpy      :", np.__version__)
print("statsmodels:", sm.__version__)

### Optional — Korean labels in plots / 그림에 한글 쓰기

Every figure in this notebook is labelled in English, so you can skip this. But Colab ships **no Korean font**, so the moment you write a Korean title yourself the characters come out as boxes. Run the cell below and it is fixed for the rest of the session — **no runtime restart needed**.

이 노트북의 그림은 전부 영문이라 건너뛰어도 됩니다. 다만 Colab에는 한글 폰트가 없어서, 여러분이 한글 제목을 쓰는 순간 네모로 깨집니다. 아래 셀을 실행하면 **런타임 재시작 없이** 바로 해결됩니다.


In [ ]:
#@title ▶ 한글 폰트 설치 / Install Korean font (optional) { display-mode: "form" }
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm

_p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
try:
    fm.fontManager.addfont(_p)                  # 캐시 재생성 없이 즉시 등록 -> 재시작 불필요
    plt.rc("font", family=fm.FontProperties(fname=_p).get_name())
    plt.rc("axes", unicode_minus=False)         # 음수 눈금 깨짐 방지
    print("Korean font ready:", plt.rcParams["font.family"][0])
except Exception as _e:
    # apt 가 막혀도 노트북 전체가 멈추지 않도록 한다. 그림 라벨만 영문/네모로 나온다.
    plt.rc("axes", unicode_minus=False)
    print("한글 폰트를 건너뜁니다 (그림 라벨이 깨질 수 있습니다):", _e)

# 함정 1. 라벨에 유니코드 마이너스(U+2212)나 공집합(U+2205)을 직접 타이핑하면
#         이 폰트에 글리프가 없어 네모로 뜹니다. ASCII 하이픈(-)을 쓰세요.
# 함정 2. 로그 축 눈금(10^-3 등)은 mathtext 로 그려지므로 unicode_minus=False 로도 막히지 않습니다.
#         로그 축을 쓸 때는 FuncFormatter 로 눈금 문자열을 직접 만드세요:
#           from matplotlib.ticker import FuncFormatter
#           ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"1e{int(round(np.log10(y)))}"))


---
## 1. Three processes, one seed

The three processes Week 2 keeps coming back to:

| Process | Definition | Stationary? |
|---|---|---|
| White noise | $w_t \sim N(0, \sigma^2)$, independent | yes — $\mu=0$, $\gamma(h)=\sigma^2 \mathbb{1}\{h=0\}$ |
| AR(1) | $x_t = \phi x_{t-1} + w_t$, $|\phi|<1$ | yes, with stationary initialization — $\gamma(h) = \phi^{|h|}\sigma^2/(1-\phi^2)$ |
| Random walk | $x_t = x_{t-1} + w_t$ | **no** — $\gamma(s,t) = \min(s,t)\sigma^2$ |

White noise and the random walk are one line each. AR(1) is the one you write.

For AR(1), draw $x_0 \sim N(0, \sigma^2/(1-\phi^2))$ independently of future innovations. This makes the simulated path stationary from the start. Starting at a fixed zero instead gives an initial transient.
Each process has a fixed random generator in the simulation cell, so re-running that cell reproduces its output.

In [ ]:
N = 500

# Reset local generators on every run. Changing one process does not advance another's RNG.
wn = np.random.default_rng(SEED).normal(0.0, 1.0, size=N)
rw = np.cumsum(np.random.default_rng(SEED + 2).normal(0.0, 1.0, size=N))

def simulate_ar1(n, phi, sigma=1.0, rng=None):
    """Return a stationary Gaussian AR(1) path; sigma is the innovation SD."""
    if not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError("n must be a positive integer.")
    if not np.isfinite(phi) or abs(phi) >= 1:
        raise ValueError("Stationary AR(1) requires abs(phi) < 1.")
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError("sigma must be finite and positive.")
    rng = np.random.default_rng() if rng is None else rng
    x = np.empty(n)
    x[0] = rng.normal(0.0, sigma / np.sqrt(1 - phi**2))
    w = rng.normal(0.0, sigma, size=n - 1)
    # TODO: loop over t = 1 .. n-1, apply the recursion, then return x.
    # Innovation for x[t] is w[t - 1]; x[0] is already initialized.
    # Remove the entire raise block below after filling the three lines.
    raise NotImplementedError(
        "Expected TODO: write the loop and return x, then remove this raise block. "
        "TODO를 채우고 이 raise 블록을 지우세요. 막히면 다음 정답 셀을 실행하세요."
    )

ar1 = simulate_ar1(N, phi=0.6, rng=np.random.default_rng(SEED + 1))
print("shapes:", wn.shape, ar1.shape, rw.shape)

In [ ]:
#@title ▶ Solution / 정답 — skip if your function works (overwrites it) { display-mode: "form" }
def simulate_ar1(n, phi, sigma=1.0, rng=None):
    """Return a stationary Gaussian AR(1) path; sigma is the innovation SD."""
    if not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError("n must be a positive integer.")
    if not np.isfinite(phi) or abs(phi) >= 1:
        raise ValueError("Stationary AR(1) requires abs(phi) < 1.")
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError("sigma must be finite and positive.")
    rng = np.random.default_rng() if rng is None else rng
    x = np.empty(n)
    x[0] = rng.normal(0.0, sigma / np.sqrt(1 - phi**2))
    w = rng.normal(0.0, sigma, size=n - 1)
    for t in range(1, n):
        x[t] = phi * x[t - 1] + w[t - 1]
    return x

ar1 = simulate_ar1(N, phi=0.6, rng=np.random.default_rng(SEED + 1))
print("ok — ar1 ready, shape", ar1.shape)

---
## 2. Look at the paths first

Before any formula: which of these has *rules that do not change with time*?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)
for ax, series, name in zip(axes, [wn, ar1, rw],
                            ["White noise", "AR(1), phi = 0.6", "Random walk"]):
    ax.plot(series, lw=0.8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel(name, fontsize=9)
axes[-1].set_xlabel("t")
fig.suptitle("Fixed seeds, three processes", y=0.98)
plt.tight_layout()
plt.show()

for name, series in [("WN", wn), ("AR1", ar1), ("RW", rw)]:
    mid = len(series) // 2
    if mid == 0:
        print(f"{name}: need at least two observations for a split comparison")
        continue
    first, second = series[:mid], series[mid:]
    print(f"{name}: first {len(first)} mean/sd = {first.mean():+.3f}/{first.std():.3f}; "
          f"last {len(second)} mean/sd = {second.mean():+.3f}/{second.std():.3f}")

Compare the printed means and standard deviations. For white noise and stationary AR(1), the population mean and variance are constant, but estimates from two finite segments need not agree. Differences can be substantial when the sample is short or dependence is strong.

These plots and split summaries are clues, **not a test or proof of stationarity**. We know which simulated process is stationary from its model and initialization.

A random walk can revisit earlier levels, but it has no restoring force toward a fixed mean. Its variance grows across independent paths at the same time index. With this code, array position 0 holds the first increment, so `rw[t]` corresponds to the mathematical $X_{t+1}$ and has variance $(t+1)\sigma^2$.

---
## 3. The sample ACF, from the formula

Slide 24 gave it:

$$\hat\gamma(h) = \frac{1}{n}\sum_{t=1}^{n-h}(x_{t+h}-\bar x)(x_t-\bar x), \qquad
\hat\rho(h) = \frac{\hat\gamma(h)}{\hat\gamma(0)}$$

The sum has only $n-h$ terms but we still divide by $n$. This matches `statsmodels` with `adjusted=False`. Other conventions exist; here we use the divide-by-n definition consistently.

In [ ]:
def sample_acf(x, max_lag=30):
    """Divide-by-n sample ACF, including lag 0."""
    x = np.asarray(x, dtype=float)
    if x.ndim != 1 or x.size < 2 or not np.all(np.isfinite(x)):
        raise ValueError("x must be a finite 1-D series with at least two observations.")
    n = len(x)
    if not isinstance(max_lag, (int, np.integer)) or not 0 <= max_lag < n:
        raise ValueError("max_lag must be an integer with 0 <= max_lag < len(x).")
    d = x - x.mean()
    if np.dot(d, d) == 0:
        raise ValueError("ACF is undefined for a constant series (zero sample variance).")
    gamma = np.empty(max_lag + 1)
    # TODO: fill gamma[h] for h = 0 .. max_lag; return gamma / gamma[0].
    # Hint: sum d[h:] * d[:n-h], then divide by n.
    # Remove the entire raise block after writing the loop and return.
    raise NotImplementedError(
        "Expected TODO: fill gamma and return gamma / gamma[0], then remove this raise block. "
        "TODO를 채우고 이 raise 블록을 지우세요. 막히면 다음 정답 셀을 실행하세요."
    )

mine = sample_acf(ar1, 20)
ref = sm.tsa.acf(ar1, nlags=20, fft=False, adjusted=False)
print("max abs difference vs statsmodels:", np.max(np.abs(mine - ref)))
assert np.allclose(mine, ref, atol=1e-10, rtol=0), "Check centering, lag slices, and divisor."
print("match ✓")

In [ ]:
#@title ▶ Solution / 정답 — skip if your function works (overwrites it) { display-mode: "form" }
def sample_acf(x, max_lag=30):
    """Divide-by-n sample ACF, including lag 0."""
    x = np.asarray(x, dtype=float)
    if x.ndim != 1 or x.size < 2 or not np.all(np.isfinite(x)):
        raise ValueError("x must be a finite 1-D series with at least two observations.")
    n = len(x)
    if not isinstance(max_lag, (int, np.integer)) or not 0 <= max_lag < n:
        raise ValueError("max_lag must be an integer with 0 <= max_lag < len(x).")
    d = x - x.mean()
    if np.dot(d, d) == 0:
        raise ValueError("ACF is undefined for a constant series (zero sample variance).")
    gamma = np.array([np.sum(d[h:] * d[:n - h]) / n for h in range(max_lag + 1)])
    return gamma / gamma[0]

mine = sample_acf(ar1, 20)
ref = sm.tsa.acf(ar1, nlags=20, fft=False, adjusted=False)
print("max abs difference vs statsmodels:", np.max(np.abs(mine - ref)))
assert np.allclose(mine, ref, atol=1e-10, rtol=0), "Check centering, lag slices, and divisor."
print("match ✓")

Now compare the sample ACFs and the stationary AR(1) population curve. Individual sample bars vary across realizations. We defer formal uncertainty bands to Week 6; the familiar $\pm2/\sqrt{n}$ approximation assumes white noise and is not a universal band for these three processes.

In [ ]:
def plot_acf_bars(ax, x, title, max_lag=30, theo=None):
    """Sample ACF as bars. Deliberately no significance band — that is a Week 6 tool."""
    r = sample_acf(x, max_lag)
    ax.bar(range(max_lag + 1), r, width=0.35, color="#3b6ea5")
    ax.axhline(0, color="k", lw=0.7)
    if theo is not None:
        ax.plot(range(max_lag + 1), theo, "o--", color="#111111", ms=3, lw=0.9, label="theoretical")
        ax.legend(fontsize=8)
    ax.set_title(title, fontsize=10); ax.set_xlabel("lag h")
    return r

lags = np.arange(31)
fig, axes = plt.subplots(3, 1, figsize=(11, 8))
r_wn = plot_acf_bars(axes[0], wn,  "White noise - no linear memory")
r_ar = plot_acf_bars(axes[1], ar1, "AR(1), phi = 0.6 - geometric decay", theo=0.6 ** lags)
r_rw = plot_acf_bars(axes[2], rw,  "Random walk - sample ACF of a nonstationary path")
plt.tight_layout(); plt.show()

for nm, r in [("white noise", r_wn), ("AR(1)", r_ar), ("random walk", r_rw)]:
    print(f"{nm:13s} rho_hat(1) = {r[1]:+.3f}   rho_hat(5) = {r[5]:+.3f}   "
          f"largest |rho_hat(h)| for h >= 1 : {np.max(np.abs(r[1:])):.3f}")

Three processes, three interpretations. Read the numbers printed above for your current settings.

- **White noise:** the population ACF is zero at nonzero lags. Sample bars fluctuate around zero.
- **Stationary AR(1):** the population ACF is $\rho(h)=0.6^{|h|}$. Sample bars approximate this curve, but can cross zero at larger lags even though the population ACF is positive.
- **Random walk:** the sample ACF can remain large across many lags. This known simulated model is nonstationary, so these bars do not estimate a stationary lag-only population ACF. A slowly decaying sample ACF alone does not prove nonstationarity.

Changing N or the seeds changes the estimates. Section 5 examines how positive autocorrelation affects uncertainty in the sample mean.

---
## 4. Time average vs ensemble average

This is the climax of Week 2 (slide 22): **stationarity does not guarantee ergodicity.**
Two questions, both answerable in code.

1. For AR(1), does averaging **along one path** reach the same number as averaging **across many paths**?
2. Is there a stationary process where it never does?

The second one is the fixed-coin process: flip one fair coin, then output that value forever.
Every $X_t$ has the same distribution (±1 with probability ½ each) and $\gamma(h) = 1$ for every $h$ —
perfectly stationary. Its time average equals that one coin value for every sample size, so it cannot converge to the ensemble mean 0. This directly proves failure of mean ergodicity; failure of a sufficient condition alone would not prove it.

In [ ]:
M, N4 = 500, 2000

# (a) AR(1), phi = 0.6 — memory that fades
paths = np.array([simulate_ar1(N4, 0.6, rng=np.random.default_rng(2_000 + s)) for s in range(M)])
ens   = paths.mean(axis=0)                              # ensemble average at each t
one   = np.cumsum(paths[0]) / np.arange(1, N4 + 1)      # running time average of ONE path

# (b) fixed coin: X_t = c for every t, c = +1 / -1 with probability 1/2.  Stationary, not ergodic.
rng4  = np.random.default_rng(SEED + 21)
c     = rng4.choice([-1.0, 1.0], size=M)
ens_c = np.full(N4, c.mean())                           # ensemble average at each t
one_c = np.full(N4, c[0])                               # the one path never moves

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
for ax, (t_avg, e_avg, ttl) in zip(
        axes, [(one,   ens,   "AR(1), phi = 0.6 — ergodic"),
               (one_c, ens_c, "Fixed coin — stationary, NOT ergodic")]):
    ax.plot(np.arange(1, N4 + 1), t_avg, lw=1.2, color="#3b6ea5", label="time average of ONE path")
    ax.plot(np.arange(1, N4 + 1), e_avg, lw=0.9, color="#d95f02", alpha=0.8, label=f"ensemble average, {M} paths")
    ax.axhline(0.0, color="k", lw=0.8, ls="--", label="true mean mu = 0")
    ax.set_title(ttl, fontsize=10); ax.set_xlabel("n"); ax.set_ylim(-1.3, 1.3)
axes[0].set_ylabel("average"); axes[0].legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

print(f"AR(1)      one-path time average at n={N4}: {one[-1]:+.4f}    ensemble average at final t: {ens[-1]:+.4f}")
print(f"fixed coin one-path time average at n={N4}: {one_c[-1]:+.4f}    ensemble average at final t: {ens_c[-1]:+.4f}")

Left panel: the running time average approaches zero in this realization. For the stationary AR(1), $\rho(h)=0.6^{|h|}$ and $\gamma(h)=0.6^{|h|}/(1-0.6^2)\to0$, giving $\operatorname{Var}(\bar X_n)\to0$: mean-square consistency of the sample mean.

The orange curve averages a fixed number M of independent paths at each time. It still fluctuates as time advances; increasing **M**, not merely the time index, reduces its Monte Carlo variance.

Right panel: the time average stays at the coin value forever, while the ensemble average is near zero. It converges to zero as the number of independent coins increases. A single fixed-coin path cannot consistently estimate the ensemble mean.

The simulation illustrates the result; the model establishes it. Mean ergodicity concerns the sample mean, and does not by itself guarantee consistent estimation of every process property.

**One more question, and it ties the whole week together.** For the fixed coin $X_t = C$ with $C \in \{-1, +1\}$ each with probability $1/2$: the *population* ACF is $\rho(h) = 1$ at every lag, because $\gamma(h) = E[C \cdot C] = 1$ for all $h$ and $\gamma(0) = \operatorname{Var}(X_t) = 1$. Yet if you hand one realized path to `sample_acf`, it raises `ValueError` — every observed value is the same number, so the sample variance is zero.

That is the correct behaviour, not a bug. **"This one path is constant" and "the random variable has zero variance" are different statements.** The process has $\gamma(0) = 1$; the path has sample variance 0. Slide 13's remark that a *deterministic* constant process has $\gamma(0) = 0$ is about the second case, not this one. Try it:

```python
coin = np.full(200, 1.0)          # one realized fixed-coin path
sample_acf(coin, 10)              # ValueError — and that is right
```

---
## 5. What positive autocorrelation costs — $\phi=0.6$ vs $\phi=0.95$

Our Gaussian AR(1) simulation uses stationary initialization. For $|\phi|<1$, its covariance tends to zero, so the sample mean is mean-square consistent. Slide 20 gave the price. The variance at a finite sample size still depends on the covariance structure:

$$\operatorname{Var}(\bar X_n)=\frac{\gamma(0)}{n}\left[1+2\sum_{h=1}^{n-1}(1-h/n)\rho(h)\right].$$

The baseline $\gamma(0)/n$ is the variance of the mean of n independent observations **with the same marginal variance**. The bracket is the variance inflation factor; it can be below 1 for some dependence structures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, phi in zip(axes, [0.6, 0.95]):
    x = simulate_ar1(400, phi=phi, rng=np.random.default_rng(SEED + 11))
    plot_acf_bars(ax, x, f"AR(1), phi = {phi}", max_lag=40, theo=phi ** np.arange(41))
plt.tight_layout(); plt.show()

N_MEAN, REPS = 400, 2000
PHIS = [0.0, 0.6, 0.95]  # Add -0.6 for the optional experiment.
print(f"Var(xbar) over {REPS} replications, n = {N_MEAN}; stationary initialization")
print(f"{'phi':>6} {'empirical':>11} {'gamma(0)/n':>12} {'emp.factor':>11} {'exact(n)':>10} {'limit':>8}")
for phi in PHIS:
    means = [simulate_ar1(N_MEAN, phi, rng=np.random.default_rng(10_000 + s)).mean()
             for s in range(REPS)]
    emp = np.var(means, ddof=1)
    indep = (1.0 / (1 - phi**2)) / N_MEAN
    h = np.arange(1, N_MEAN)
    exact = 1 + 2 * np.sum((1 - h / N_MEAN) * phi**h)
    print(f"{phi:>6.2f} {emp:>11.5f} {indep:>12.5f} {emp/indep:>10.2f} "
          f"{exact:>10.2f} {(1+phi)/(1-phi):>8.2f}")

For the default n = 400, the **exact finite-n** inflation factors are approximately 3.98 at $\phi=0.6$ and 37.10 at $\phi=0.95$. Their large-n limits are 4 and 39. The empirical column estimates these finite-n values using 2000 independent replications; Monte Carlo error prevents an exact match.

At $\phi=0.95$, the variance-based effective sample size for estimating the mean is about $400/37.10=10.8$, relative to independent observations with the same marginal variance. Persistent **positive** autocorrelation reduces effective sample size in these examples. Other dependence patterns can help instead.

All paths now start in the stationary distribution, so there is no zero-start transient to explain. The remaining distinction is finite-n theory versus its large-n limit, plus simulation error. Mean ergodicity guarantees mean-square convergence, but does not promise that convergence is fast.

---
## 6. Your turn

Do at least two experiments. Add your code and one or two sentences below each result.

1. Generate a new path with `simulate_ar1(N, phi=-0.6, rng=np.random.default_rng(SEED + 1))`. Plot its sample ACF and the theoretical curve `(-0.6) ** np.arange(31)` using `plot_acf_bars`. Explain the alternating signs. **Optional:** add `-0.6` to `PHIS` in section 5 and re-run that cell. Can dependence lower the variance of the sample mean?
2. Simulate MA(1), $X_t=w_t+\theta w_{t-1}$ with $\theta=0.8$. Generate N+1 independent shocks and combine adjacent slices. Its theoretical ACF is zero beyond lag 1 — the cut-off is what separates MA from AR (slide 16); check whether the sample bars fluctuate around zero there. They need not be exactly zero.
3. Difference the random walk once with `np.diff(rw)`. What should the nonzero-lag ACF look like, and does the sample support that expectation?
4. Replace the AR(1) paths in section 4 with independent random walks, generated with `np.cumsum`. Does the one-path time average appear to settle? Use $\gamma(s,t)=\min(s,t)\sigma^2$ to explain why its variance does not vanish. Remember that a fixed-M ensemble average also becomes noisier as t increases for a random walk.

For sample-size sensitivity, set N = 50 and then N = 1000 in section 1 and re-run sections 1–3. Compare the estimates; do not treat visual differences alone as a stationarity test.

In [ ]:
# Scratch space for section 6.


---
## 7. Environment record

Nothing to submit this week, but from Assignment 1 onward every submission ends with this cell — it is how
a marker reproduces your numbers. Run it once now so it is not new later.

In [ ]:
import sys, platform, datetime
try:
    import torch; gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
except Exception:
    gpu = "torch not loaded"

print("run at      :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("python      :", sys.version.split()[0], "on", platform.system())
print("runtime     :", gpu)
print("seed        :", SEED)
print("numpy       :", np.__version__)
print("statsmodels :", sm.__version__)